# GeneFlow AI: clasificación taxonómica jerárquica de secuencias de ADN

Este notebook es el cuaderno de trabajo del TFM. Aquí se explora el dataset y, más adelante, se entrenan y evalúan los modelos. **La lógica reutilizable no vive aquí, sino en el paquete `taxonomy_classifier`**, que tiene tests, tipado estricto y un 90 % de cobertura mínima. El notebook solo lo importa y lo usa.

## Requisitos previos

Desde la raíz del repositorio:

| Paso | Comando | Qué hace |
|---|---|---|
| 1 | `uv sync` | Instala el paquete en modo editable junto con los grupos `dev` y `notebooks` |
| 2 | `uv run geneflow prepare` | Descarga las cuatro fuentes (~470 MB) y construye el dataset (~2-3 min) |
| 3 | `uv run jupyter lab` | Abre Jupyter. En PyCharm o VS Code, selecciona `.venv` como intérprete |

## De dónde salen los datos

Todas las fuentes contienen el gen del ARN ribosomal de la subunidad pequeña (16S en procariotas, 18S en eucariotas). **GTDB y PR2 son la taxonomía de referencia**: el resto de fuentes se traduce a ellas.

| Fuente | Aporta | Cómo se etiqueta |
|---|---|---|
| **GTDB r232** | 16S de bacterias y arqueas extraídos de genomas completos | Etiqueta propia, hasta especie |
| **PR2 5.1.1** | 18S nuclear de eucariotas, curado a mano | Etiqueta propia, hasta especie. Supergrupo y subdivisión quedan solo en el linaje original |
| **RefSeq 16S** | 16S de cepas tipo | Se traduce a GTDB por especie o, si no existe, por género |
| **SILVA 144 NR99** | Diversidad ambiental de los tres dominios | Se traduce por el nombre más profundo que exista sin ambigüedad en GTDB o PR2. La especie siempre queda nula |

## Cómo se construye

1. **Lectura y filtrado.** Se descartan las secuencias de menos de 900 pb o de más de 4000 pb, las que tienen más de un 1 % de bases ambiguas, los orgánulos y los registros que no son del gen buscado.
2. **Eliminación de duplicados.** Las secuencias idénticas se fusionan. Si llegan con etiquetas distintas, se quedan con su ancestro común más profundo y se marcan con `label_conflict`. Si ni siquiera coincide el dominio, la secuencia se descarta.
3. **División determinista**, con una semilla fija:

| `split` | Contenido |
|---|---|
| `train` | ~90 % de las secuencias |
| `val`, `test` | ~5 % cada uno, elegidos al azar por secuencia |
| `test_novel_genus` | Todas las secuencias de ~5 % de los géneros, que el modelo nunca ve al entrenar. Mide si sabe generalizar hasta familia ante un género nuevo |

`build_report.json` detalla, para cada fuente, cuántas secuencias se leyeron, cuántas se descartaron y por qué, y hasta qué rango se etiquetaron. También recoge las estadísticas de la fusión y la configuración usada.

**El equilibrado de clases y el data augmentation no se guardan en disco.** Se aplican al vuelo durante el entrenamiento con `taxonomy_classifier.training`:
- `balanced_weights` da pesos de muestreo;
- `SequenceAugmenter` introduce mutaciones y recortes.

Así no se filtran copias de test al entrenamiento y la evaluación se hace siempre sobre secuencias reales.

## Qué hace la celda de configuración

| Variable | Contenido |
|---|---|
| `PROJECT_ROOT` | La raíz del repositorio, detectada buscando `pyproject.toml`. El notebook funciona desde cualquier directorio |
| `LAYOUT` | Las rutas de datos: `LAYOUT.raw_dir(fuente)`, `LAYOUT.interim_dir`, `LAYOUT.processed_dir` |
| `DATASET_PATH`, `REPORT_PATH` | El Parquet final y su informe |
| `FIGURES_DIR` | `reports/figures/`: figuras exportadas para la memoria (se crea si no existe) |
| `SOURCES` | Las fuentes de datos, indexadas por su nombre corto (`gtdb_r232`, `pr2_5.1.1`…) |
| `RANK_COLUMNS` | Las columnas taxonómicas en orden jerárquico |
| `SPLITS` | Los nombres de las cuatro particiones |
| `KINGDOMS` | Los seis reinos clásicos |
| `dataset` | Un `LazyFrame` de Polars sobre el Parquet. No carga nada en memoria hasta `.collect()` |
| `report` | El informe de construcción, ya leído como diccionario |

Además configura `%autoreload`, el `logging`, cómo muestra Polars las tablas y el estilo de Matplotlib (figuras a 300 ppp). Si el dataset no existe, la celda se detiene con el comando que hay que ejecutar.

## Estructura del dataset

| Columna | Tipo | Descripción |
|---|---|---|
| `seq_hash` | binario | Huella BLAKE2b de 16 bytes de la secuencia. Identifica cada fila |
| `sequence` | texto | Secuencia de ADN normalizada (`U` pasa a `T`, en mayúsculas) |
| `length`, `n_ambiguous` | entero | Longitud y número de bases distintas de `A`, `C`, `G` y `T` |
| `domain` … `species` | texto o nulo | Un rango por columna. Nulo si se desconoce, es de relleno o hubo conflicto entre fuentes |
| `kingdom` | texto o nulo | Reino clásico: `Bacteria`, `Archaea`, `Animalia`, `Fungi`, `Plantae` o `Protista`. Nulo si no se puede determinar o las fuentes discrepan |
| `sources` | lista de texto | Fuentes que aportaron la secuencia (`gtdb`, `pr2`, `refseq`, `silva`) |
| `accessions` | lista de texto | Identificadores originales, como `fuente:accesión` |
| `n_records` | entero | Cuántos registros se fusionaron en esta fila |
| `label_conflict` | booleano | Las fuentes discrepaban y la etiqueta se recortó a su ancestro común |
| `split` | texto | `train`, `val`, `test` o `test_novel_genus` |

**Cómo se asigna `kingdom`:**
- **GTDB y RefSeq** usan el dominio.
- **PR2** usa su linaje original: la subdivisión Metazoa es `Animalia`, la subdivisión Fungi es `Fungi`, las divisiones Streptophyta, Chlorophyta, Prasinodermophyta, Rhodophyta y Glaucophyta son `Plantae` (Archaeplastida fotosintéticas, algas rojas incluidas) y el resto es `Protista`.
- **SILVA**: en procariotas usa el dominio. En eucariotas hereda el reino de su traducción a PR2, solo si no es ambiguo.

`Protista` no es un grupo natural: reúne todos los eucariotas que no son animales, plantas ni hongos.

En eucariotas, las columnas siguen la jerarquía de PR2. Por ejemplo, en un hongo `phylum` = Opisthokonta y `class` = Ascomycota. Es coherente dentro de PR2, pero no coincide con la nomenclatura clásica.


In [1]:
%load_ext autoreload
%autoreload 2

import json
import logging
from pathlib import Path

import matplotlib.pyplot as plt
import polars as pl

from taxonomy_classifier.data.build import DataLayout
from taxonomy_classifier.data.columns import RANK_COLUMNS
from taxonomy_classifier.data.kingdom import Kingdom
from taxonomy_classifier.data.sources.registry import DEFAULT_SOURCES
from taxonomy_classifier.data.split import Split

PROJECT_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents) if (path / "pyproject.toml").exists()
)

LAYOUT = DataLayout(root=PROJECT_ROOT / "data")
DATASET_PATH = LAYOUT.dataset_path
REPORT_PATH = LAYOUT.report_path
FIGURES_DIR = PROJECT_ROOT / "reports" / "figures"

SOURCES = {source.slug: source for source in DEFAULT_SOURCES}
SPLITS = [split.value for split in Split]
KINGDOMS = [kingdom.value for kingdom in Kingdom]

if not DATASET_PATH.exists():
    msg = f"{DATASET_PATH} not found; run 'uv run geneflow prepare' from {PROJECT_ROOT}"
    raise FileNotFoundError(msg)

FIGURES_DIR.mkdir(parents=True, exist_ok=True)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)-8s %(name)s: %(message)s",
)

pl.Config.set_tbl_rows(20)
pl.Config.set_tbl_cols(-1)
pl.Config.set_tbl_width_chars(200)
pl.Config.set_fmt_str_lengths(80)
pl.Config.set_thousands_separator(".")
pl.Config.set_decimal_separator(",")

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update(
    {
        "figure.figsize": (10, 5),
        "figure.dpi": 110,
        "savefig.dpi": 300,
        "savefig.bbox": "tight",
        "axes.titleweight": "bold",
    }
)

dataset = pl.scan_parquet(DATASET_PATH)
report = json.loads(REPORT_PATH.read_text(encoding="utf-8"))